### Tranform Customer Data
- Remove records with NULL Customer_id
- Remove exact duplicate records
- Remove duplicate records based on created_timestamp
- CAST the column to the correct data type
- Write transformed data to silver schema

###1. Remove records with NULL Customer_id

In [0]:
dfCustomers = spark.table("gizmobox_sivan.bronze.v_customers")
display(dfCustomers)

In [0]:
df_Valid_Customers = dfCustomers.filter('customer_id is not null')
display(df_Valid_Customers)

In [0]:
df_Valid_Customers = dfCustomers.filter(dfCustomers.customer_id.isNotNull())
display(df_Valid_Customers)

###2. Remove exact duplicate records

In [0]:
df_distinct_Valid_Customers = dfCustomers.filter(dfCustomers.customer_id.isNotNull()).distinct()
display(df_distinct_Valid_Customers)

In [0]:
%sql
select distinct *
from gizmobox_sivan.bronze.v_customers
where customer_id is not null

###3. Remove duplicate records based on created_timestamp

In [0]:
from pyspark.sql.functions import max, min
df_max_customerId = df_distinct_Valid_Customers.groupBy('customer_id').agg(max('created_timestamp').alias('max_created_timestamp'))
display(df_max_customerId)

In [0]:
from pyspark.sql import functions as f

df_Valid_CustomersData = (
        df_distinct_Valid_Customers.join(df_max_customerId, 
                (df_distinct_Valid_Customers.customer_id ==  df_max_customerId.customer_id) & 
                (df_distinct_Valid_Customers.created_timestamp == df_max_customerId.max_created_timestamp)
                , "inner").select(df_distinct_Valid_Customers['*'])
) 
                                                          
                                                        
display(df_Valid_CustomersData)



###4. CAST the column to the correct data type

In [0]:
df_CS_AfterCasting = (
    df_Valid_CustomersData.select(
        df_Valid_CustomersData.created_timestamp.cast('timestamp').alias('created_timestamp'),
        df_Valid_CustomersData.customer_id.cast('long').alias('customer_id'),
        df_Valid_CustomersData.customer_name.cast('string').alias('customer_name'),
        df_Valid_CustomersData.date_of_birth.cast('date').alias('date_of_birth'),
        df_Valid_CustomersData.email.cast('string').alias('email'),
        df_Valid_CustomersData.member_since.cast('date').alias('member_since'),
        df_Valid_CustomersData.telephone.cast('string').alias('telephone'),
        df_Valid_CustomersData.file_path.cast('string').alias('file_path'),
    )
)

display(df_CS_AfterCasting)

### 5. Write data to a Delta table

In [0]:
df_CS_AfterCasting.writeTo("gizmobox_sivan.silver.py_customers").createOrReplace()

In [0]:
df = spark.table("gizmobox_sivan.silver.py_customers")
display(df)

- dfCustomers = spark.table("gizmobox_sivan.bronze.v_customers")
- display(dfCustomers)
- Remove records with NULL Customer_id
- Remove exact duplicate records
- Remove duplicate records based on created_timestamp
- CAST the column to the correct data type

In [0]:
#Practice***********

from pyspark.sql.functions import max

dfCustomers = spark.table("gizmobox_sivan.bronze.v_customers")

dfvalidCust = dfCustomers.filter(dfCustomers.customer_id.isNotNull())

dfdistinctCust = dfvalidCust.distinct()

dfMaxCustomer = dfdistinctCust.groupBy("customer_id").agg(max("created_timestamp").alias("max_created_timestamp"))

dfDistinctCusomers = dfdistinctCust.join(dfMaxCustomer, (dfdistinctCust.customer_id == dfMaxCustomer.customer_id) & (dfdistinctCust.created_timestamp == dfMaxCustomer.max_created_timestamp),"inner").select(dfdistinctCust["*"])

dffinalCustomerList = dfDistinctCusomers.select(
        dfDistinctCusomers.created_timestamp.cast('timestamp').alias('created_timestamp'),
        dfDistinctCusomers.customer_id.cast('long').alias('customer_id'),
        dfDistinctCusomers.customer_name.cast('string').alias('customer_name'),
        dfDistinctCusomers.date_of_birth.cast('date').alias('date_of_birth'),
        dfDistinctCusomers.email.cast('string').alias('email'),
        dfDistinctCusomers.member_since.cast('date').alias('member_since'),
        dfDistinctCusomers.telephone.cast('string').alias('telephone'),
        dfDistinctCusomers.file_path.cast('string').alias('file_path'),
)

display(dffinalCustomerList)